# 10 Statistik LMS


In [ ]:
# Jalur REVISED dan pembaca hasil praproses.
from pathlib import Path
import os, sys
from IPython.display import display
try:
    from google.colab import drive
except ImportError:
    pass
else:
    if not Path('/content/drive/MyDrive').is_dir():drive.mount('/content/drive',force_remount=False)
_explicit=os.environ.get('CARE_KM_ROOT') or os.environ.get('CARE_KM_PAPER_ROOT')
_options=([Path(_explicit)] if _explicit else []) + [Path('/content/drive/MyDrive/REVISED'),
    Path(__file__).resolve().parent.parent if '__file__' in globals() else Path.cwd(),Path.cwd(),Path.cwd().parent]
_ROOT_REVISED=next((p.resolve() for p in _options if all((p/d).is_dir() for d in ('DATA','LIB','PYTHON'))),None)
if _ROOT_REVISED is None:raise FileNotFoundError('Letakkan DATA, LIB, PYTHON di MyDrive/REVISED.')
sys.path.insert(0,str(_ROOT_REVISED/'LIB'))
# Jangan memakai modul versi lama yang masih tersimpan di memori Colab.
import importlib
_loaded=sys.modules.get('pipeline_paths')
if _loaded is None or (getattr(_loaded,'PIPELINE_LAYOUT_VERSION',None)!='2026.11.06-experiments-3'
    or Path(getattr(_loaded,'__file__','')).resolve()!=(_ROOT_REVISED/'LIB/pipeline_paths.py').resolve()):
    for _mod in ('pipeline_paths','pipeline_data','pipeline_runner','pipeline_cache','experiment_lib','care_km','generalization_data'):
        sys.modules.pop(_mod,None)
    importlib.invalidate_caches()
from pipeline_paths import configure,stage_dir,lms_data,pendikom_data,artifact_path,record_resample
configure(_ROOT_REVISED)


In [ ]:
from pipeline_data import preflight
preflight(_ROOT_REVISED,benchmarks=False)


In [ ]:
from pathlib import Path
import json
import os
import sys
import time
import hashlib

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception:
    pass

# Find paper root from the standard Drive path or from the local package.
_candidates = [
    Path(os.environ.get('CARE_KM_PAPER_ROOT', '')) if os.environ.get('CARE_KM_PAPER_ROOT') else None,
    _ROOT_REVISED,
    _ROOT_REVISED,
    Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent,
]
PAPER_ROOT = next((p.resolve() for p in _candidates if p is not None and (p/'LIB').is_dir() and (p/'DATA').is_dir()), None)
if PAPER_ROOT is None:
    raise FileNotFoundError('Root REVISED tidak ditemukan.')
LIB_DIR = PAPER_ROOT / 'LIB'
DATA_DIR = PAPER_ROOT / 'DATA'
OUT_DIR = stage_dir('10_Statistik_LMS')
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(LIB_DIR))

import experiment_lib as el
import care_km as ck

CARE_CONFIG = dict(el.FROZEN_CARE_CONFIG)
K = el.K
METHODS = list(el.METHODS)
print('PAPER_ROOT:', PAPER_ROOT)
print('CARE config:', CARE_CONFIG)


from scipy.stats import wilcoxon, rankdata

RAW_FILE = artifact_path('06_robustness_raw.csv','10_Statistik_LMS')
if not RAW_FILE.exists():
    raise FileNotFoundError('Jalankan 09_Robustness_LMS_30x80.ipynb terlebih dahulu.')
raw = pd.read_csv(RAW_FILE)

PRIMARY = {
    'initial_cost': 'lower',
    'final_cost': 'lower',
    'dbi': 'lower',
    'si': 'higher',
}
BASELINES = [m for m in METHODS if m != 'CARE-KM']
rows = []

for baseline in BASELINES:
    care = raw[raw['method']=='CARE-KM'].set_index('rep').sort_index()
    ref = raw[raw['method']==baseline].set_index('rep').sort_index()
    common = care.index.intersection(ref.index)
    if len(common) != 30:
        raise RuntimeError(f'{baseline}: paired reps = {len(common)}, expected 30')
    if not (care.loc[common,'sample_hash'].astype(str).to_numpy() == ref.loc[common,'sample_hash'].astype(str).to_numpy()).all():
        raise RuntimeError(f'{baseline}: sample hashes tidak matched')

    for metric, direction in PRIMARY.items():
        a = care.loc[common, metric].astype(float).to_numpy()
        b = ref.loc[common, metric].astype(float).to_numpy()
        improvement = (b - a) if direction == 'lower' else (a - b)
        nz = improvement[np.abs(improvement) > 1e-12]

        if len(nz) == 0:
            p_raw = 1.0
            rbc = 0.0
        else:
            try:
                p_raw = float(wilcoxon(nz, alternative='greater', zero_method='wilcox', method='auto').pvalue)
            except ValueError:
                p_raw = 1.0
            ranks = rankdata(np.abs(nz), method='average')
            w_plus = float(ranks[nz > 0].sum())
            w_minus = float(ranks[nz < 0].sum())
            rbc = (w_plus - w_minus) / (w_plus + w_minus) if (w_plus + w_minus) else 0.0

        wins = int((improvement > 1e-12).sum())
        ties = int((np.abs(improvement) <= 1e-12).sum())
        losses = int((improvement < -1e-12).sum())

        rng = np.random.default_rng(700000 + 97*BASELINES.index(baseline) + list(PRIMARY).index(metric))
        boot = np.empty(5000)
        for i in range(len(boot)):
            boot[i] = rng.choice(improvement, size=len(improvement), replace=True).mean()
        lo, hi = np.quantile(boot, [0.025,0.975])

        rows.append({
            'baseline': baseline, 'metric': metric, 'direction': direction,
            'n_pairs': len(common),
            'care_mean': float(a.mean()), 'baseline_mean': float(b.mean()),
            'mean_improvement': float(improvement.mean()),
            'wins': wins, 'ties': ties, 'losses': losses,
            'win_rate': float(wins/len(improvement)),
            'rank_biserial': float(rbc),
            'bootstrap_lo': float(lo), 'bootstrap_hi': float(hi),
            'p_raw': p_raw,
        })

stats = pd.DataFrame(rows)
stats['p_holm'] = el.holm_adjust(stats['p_raw'].to_numpy())
stats['significant_0_05'] = stats['p_holm'] < 0.05

el.save_csv(stats, artifact_path('07_wilcoxon_holm.csv','10_Statistik_LMS'))
el.save_csv(stats[['baseline','metric','mean_improvement','bootstrap_lo','bootstrap_hi']], artifact_path('07_bootstrap_ci.csv','10_Statistik_LMS'))
el.save_csv(stats[['baseline','metric','wins','ties','losses','win_rate','rank_biserial']], artifact_path('07_effect_size_wtl.csv','10_Statistik_LMS'))

# Efficiency is reported transparently as paired descriptive difference; no superiority direction is imposed.
eff = []
for baseline in BASELINES:
    care = raw[raw['method']=='CARE-KM'].set_index('rep').sort_index()
    ref = raw[raw['method']==baseline].set_index('rep').sort_index()
    common = care.index.intersection(ref.index)
    for metric in ['init_sec','cluster_sec','total_sec','iterations']:
        d = care.loc[common, metric].astype(float).to_numpy() - ref.loc[common, metric].astype(float).to_numpy()
        eff.append({
            'baseline': baseline, 'metric': metric,
            'care_mean': float(care.loc[common,metric].astype(float).mean()),
            'baseline_mean': float(ref.loc[common,metric].astype(float).mean()),
            'care_minus_baseline_mean': float(d.mean()),
            'care_faster_or_lower_rate': float((d < 0).mean()),
        })
eff = pd.DataFrame(eff)
el.save_csv(eff, artifact_path('07_efficiency_paired_summary.csv','10_Statistik_LMS'))

display(stats.sort_values(['metric','p_holm']))
display(eff)